In [ ]:
#TODO
# SAM, MedSAM(fine-tuned decoder), SAMMed2D(fine-tuned all) results over 51 datasets

#### How can we fine tune SAM efficitively with the least data?  

1. Will different selection query give us diferent results?    
constraint:  
 dataset size:51 (40 train; 11 test)  
 epochs: 1000 for add one sample 
 fixed random seed  
 model: SAM  
 checkpoint: b  
 fine-tuning structure: mask decoder  

* baseline: random selection from 40 dataset
* entropy with dropout 
* entropy with dropout + artifacts
* encoder of SAM + clustering  



- Plot: Accuracy on the 11 test set as a function of number of selected samples (training)
- ID of selected samples for each iteration

In [1]:
import datasets.path as path
# import nibabel as nib
from glob import glob
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import monai
# from utils.SurfaceDice import compute_dice_coefficient
from tqdm import tqdm
import json
from segment_anything import SamPredictor, sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
import matplotlib.pyplot as plt

In [2]:
# training and sampling dataset path prefix
prefix = './datasets/RAINE_organ_51'
task = 'MRI_Pancreas'
training_pool_path = os.path.join(prefix, task, 'train')
# label id
# left kidney: 1,
# right kidney: 2,
# pancreas: 3,
# background: 0,
label_id = 3

#SAM MODEL TYPE 
sam_model_type = 'vit_b'
# SAM checkpoint
checkpoint = './checkpoints/SAM/sam_vit_b_01ec64.pth'
# device 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# sample strategy
strategy = 'random'
# active learning fine tunning checkpoint
base_model = 'SAM'
save_path_ckp = os.path.join('./checkpoints/', base_model + '_' + task + '_' + strategy)
os.makedirs(save_path_ckp, exist_ok=True)
# sampling dataset path
sampling_datapath = os.path.join(prefix, task, base_model+'_sampling')
os.makedirs(sampling_datapath, exist_ok=True)

In [3]:
#%% create a dataset class to load npz data and return back image embeddings and ground truth
class NpzDataset(Dataset): 
    def __init__(self, sample_pool_path):
        self.npz_files = sorted(sample_pool_path) 
        # print(self.npz_files[0])
        self.npz_data = [np.load(f) for f in self.npz_files]
        self.ori_gts = np.vstack([d['gts'] for d in self.npz_data])
        self.img_embeddings = np.vstack([d['img_embeddings'] for d in self.npz_data])
        # print(f"{self.img_embeddings.shape=}, {self.ori_gts.shape=}")
    
    def __len__(self):
        return self.ori_gts.shape[0]

    def __getitem__(self, index):
        img_embed = self.img_embeddings[index]
        gt2D = self.ori_gts[index]
        y_indices, x_indices = np.where(gt2D > 0)
        x_min, x_max = np.min(x_indices), np.max(x_indices)
        y_min, y_max = np.min(y_indices), np.max(y_indices)
        # add perturbation to bounding box coordinates
        # H, W = gt2D.shape
        # x_min = max(0, x_min - np.random.randint(0, 20))
        # x_max = min(W, x_max + np.random.randint(0, 20))
        # y_min = max(0, y_min - np.random.randint(0, 20))
        # y_max = min(H, y_max + np.random.randint(0, 20))
        # bboxes = np.array([x_min, y_min, x_max, y_max])
        # whole image as bbox
        bboxes = np.array([0, 0, gt2D.shape[1], gt2D.shape[0]])
        # convert img embedding, mask, bounding box to torch tensor
        return torch.tensor(img_embed).float(), torch.tensor(gt2D[None, :,:]).long(), torch.tensor(bboxes).float()

In [5]:
training_pool = glob(os.path.join(training_pool_path, '*.npz'))
sample_pool = []
num_epochs = 1
batch_size = 16 # make sure batch size is smaller than sample slices
losses = []
best_loss = 1e10
sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)
sam_model.train()
# Set up the optimizer, hyperparameter tuning will improve performance here
optimizer = torch.optim.Adam(sam_model.mask_decoder.parameters(), lr=1e-5, weight_decay=0)
seg_loss = monai.losses.DiceCELoss(sigmoid=True, squared_pred=True, reduction='mean')

for i in range(len(training_pool)):
    np.random.seed(2023)
    next_sample = np.random.choice(training_pool)
    sample_pool.append(next_sample)
    training_pool.remove(next_sample)

    # next_sample = np.random.choice(training_pool)
    # sample_pool.append(next_sample)
    # training_pool.remove(next_sample)
    num_samples = len(sample_pool)
    if num_samples > 2 and num_samples <= 5:
        batch_size = 32
    elif num_samples > 5:
        batch_size = 64
    
    print('Number of samples: ', num_samples, '\tBatch size: ', batch_size)
    sample_dataset = NpzDataset(sample_pool)
    sample_dataloader = DataLoader(sample_dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(num_epochs):
        epoch_loss = 0
        for step, (image_embedding, gt2D, boxes) in enumerate(tqdm(sample_dataloader)):
            # img_embed: (B, 256, 64, 64), gt2D: (B, 1, 256, 256), bboxes: (B, 4)
            # print(f"{image_embedding.shape=}, {gt2D.shape=}, {boxes.shape=}")
            with torch.no_grad():
                box_np = boxes.numpy() # [0, 0, 256, 256]
                sam_trans = ResizeLongestSide(sam_model.image_encoder.img_size)
                box = sam_trans.apply_boxes(box_np, (gt2D.shape[-2], gt2D.shape[-1]))
                box_torch = torch.as_tensor(box, dtype=torch.float, device=device)
                if len(box_torch.shape) == 2:
                    box_torch = box_torch[:, None, :] # (B, 1, 4)
                # get prompt embeddings 
                sparse_embeddings, dense_embeddings = sam_model.prompt_encoder(
                    points=None,
                    boxes=box_torch,
                    masks=None,
                )
            # predicted masks
            # print(f"{image_embedding.shape=}, {sparse_embeddings.shape=}, {dense_embeddings.shape=}")
            mask_predictions, _ = sam_model.mask_decoder(
                image_embeddings=image_embedding.to(device), # (B, 256, 64, 64)
                image_pe=sam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
                sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
                dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
                multimask_output=False,
            )

            loss = seg_loss(mask_predictions, gt2D.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= step
        losses.append(epoch_loss)
        print(f'EPOCH: {epoch}, Loss: {epoch_loss}')
        # save the latest model checkpoint
        torch.save(sam_model.state_dict(), os.path.join(save_path_ckp, f'sam_{sam_model_type}_{num_samples:02d}_latest.pth'))
        # save the best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(sam_model.state_dict(), os.path.join(save_path_ckp, f'sam_{sam_model_type}_{num_samples:02d}_best.pth'))

    # plot loss
    plt.plot(losses)
    plt.title('Dice + Cross Entropy Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    # plt.show() # comment this line if you are running on a server
    plt.savefig(os.path.join(save_path_ckp, f'sam_{sam_model_type}_{num_samples:02d}_train_loss.png'))
    plt.close()

with open(os.path.join(sampling_datapath, f'{strategy}_pool.json'), 'w') as f:
    json.dump(sample_pool, f, indent=4)


Number of samples:  1 	Batch size:  16


100%|██████████| 3/3 [00:00<00:00,  5.15it/s]


EPOCH: 0, Loss: 1.4927488565444946
Number of samples:  2 	Batch size:  16


100%|██████████| 4/4 [00:01<00:00,  3.91it/s]


EPOCH: 0, Loss: 1.3181916077931721
Number of samples:  3 	Batch size:  32


...

EPOCH: 0, Loss: 0.2480649329148806
Number of samples:  39 	Batch size:  32


100%|██████████| 41/41 [00:15<00:00,  2.61it/s]


EPOCH: 0, Loss: 0.24840974546968936
Number of samples:  40 	Batch size:  32


100%|██████████| 43/43 [00:16<00:00,  2.65it/s]


EPOCH: 0, Loss: 0.2436043613013767


In [34]:
def get_bbox_from_mask(mask):
    '''Returns a bounding box from a mask'''
    # y_indices, x_indices = np.where(mask > 0)
    # x_min, x_max = np.min(x_indices), np.max(x_indices)
    # y_min, y_max = np.min(y_indices), np.max(y_indices)
    # # add perturbation to bounding box coordinates
    # H, W = mask.shape
    # np.random.seed(2023)
    # x_min = max(0, x_min - np.random.randint(0, 20))
    # x_max = min(W, x_max + np.random.randint(0, 20))
    # y_min = max(0, y_min - np.random.randint(0, 20))
    # y_max = min(H, y_max + np.random.randint(0, 20))
    bbox = np.array([0, 0, mask.shape[1], mask.shape[0]])
    # return np.array([x_min, y_min, x_max, y_max])
    return bbox

def compute_dice_coefficient(mask_gt, mask_pred):
  """Compute soerensen-dice coefficient.

  compute the soerensen-dice coefficient between the ground truth mask `mask_gt`
  and the predicted mask `mask_pred`. 
  
  Args:
    mask_gt: 3-dim Numpy array of type bool. The ground truth mask.
    mask_pred: 3-dim Numpy array of type bool. The predicted mask.

  Returns:
    the dice coeffcient as float. If both masks are empty, the result is NaN
  """
  volume_sum = mask_gt.sum() + mask_pred.sum()
  if volume_sum == 0:
    return np.NaN
  volume_intersect = (mask_gt & mask_pred).sum()
  return 2*volume_intersect / volume_sum
 

In [32]:
def infer(imgs, ckp_path, model_type, device):
    # infer on fine-tuned sam model
    medsam_segs = []
    bboxes = []
    for img in imgs:
        # bbox = get_bbox_from_mask(gt)
        bbox = np.array([0 ,0, img.shape[0], img.shape[1]])
        bboxes.append(bbox)
        
        # predict the segmentation mask using the fine-tuned model
        sam_model = sam_model_registry[model_type](checkpoint=ckp_path).to(device)

        sam_trans = ResizeLongestSide(sam_model.image_encoder.img_size)
        H, W = img.shape[:2]
        resize_img = sam_trans.apply_image(img)
        resize_img_tensor = torch.as_tensor(resize_img.transpose(2, 0, 1)).to(device)
        input_image = sam_model.preprocess(resize_img_tensor[None,:,:,:]) # (1, 3, 1024, 1024)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(input_image.to(device)) # (1, 256, 64, 64)
            # convert box to 1024x1024 grid
            bbox = sam_trans.apply_boxes(bbox, (H, W))
            box_torch = torch.as_tensor(bbox, dtype=torch.float, device=device)
            if len(box_torch.shape) == 2:
                box_torch = box_torch[:, None, :] # (B, 1, 4)
            
            sparse_embeddings, dense_embeddings = sam_model.prompt_encoder(
                points=None,
                boxes=box_torch,
                masks=None,
            )
            medsam_seg_prob, _ = sam_model.mask_decoder(
                image_embeddings=image_embedding.to(device), # (B, 256, 64, 64)
                image_pe=sam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
                sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
                dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
                multimask_output=False,
                )
            medsam_seg_prob = torch.sigmoid(medsam_seg_prob)
            # convert soft mask to hard mask
            medsam_seg_prob = medsam_seg_prob.cpu().numpy().squeeze()
            medsam_seg = (medsam_seg_prob > 0.5).astype(np.uint8)
            medsam_segs.append(medsam_seg)
    return np.stack(medsam_segs, axis=0), bboxes

In [35]:
ckp_paths = sorted(glob(os.path.join(save_path_ckp, '*best.pth')))
testing_pool_path = os.path.join(prefix, task, 'test')
testing_pool = glob(os.path.join(testing_pool_path, '*.npz'))
dice_test = []

for p in tqdm(ckp_paths):
    avg_dice = []
    for t in testing_pool:
        imgs, gts = np.load(t)['imgs'], np.load(t)['gts']
        pre, _ = infer(imgs, p, sam_model_type, device)
        dice = compute_dice_coefficient(gts, pre)
        avg_dice.append(dice)
    dice_test.append(np.mean(avg_dice))
    

 19%|█▉        | 7/37 [44:39<3:11:07, 382.26s/it]